# Spotter — Final USDA Food Catalog

This notebook takes the **processed output from the first USDA normalization notebook** and produces the final CSV intended for the Spotter `foods` table.

## Input

Expected:

```text
data/
└── processed/
    ├── foods_for_db.csv
    └── usda_foods_stage.csv
```

`foods_for_db.csv` is the main input.  
`usda_foods_stage.csv` is used only for audit/debug information when available.

## Final output

```text
data/final/foods_final.csv
```

This is the CSV to import into PostgreSQL.

The notebook also writes audit files so every automatic decision can be inspected:

```text
data/final/audit/
├── dropped_exact_name_duplicates.csv
├── dropped_mixed_dishes.csv
├── dropped_invalid_rows.csv
└── duplicate_conflicts.csv
```

---

## Final catalog policy

1. Keep only rows that match the Spotter schema.
2. Reject impossible/missing nutrition values.
3. By default, remove `MIXED_DISHES` because Spotter has a separate `Recipe` entity.
4. Normalize food names only for **comparison**; the displayed `name_en` is preserved.
5. If multiple USDA rows have the same normalized English name, keep one using this source priority:

```text
Foundation > FNDDS > SR Legacy
```

6. If duplicate rows disagree substantially on nutrition, we still select the preferred source deterministically, but also write the whole group to `duplicate_conflicts.csv` for human inspection.
7. Enforce uniqueness of `(source, external_source_id)`.
8. Do not create database IDs or timestamps; Prisma/PostgreSQL owns those fields.


In [ ]:
from pathlib import Path
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)


# ---------------------------------------------------------
# FIND PROJECT ROOT
# ---------------------------------------------------------

def find_project_root(start: Path):
    start = start.resolve()

    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find data/processed from the current directory or any parent. "
        "Run this notebook from inside your Spotter-EDA project."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FINAL_DIR = PROJECT_ROOT / "data" / "final"
AUDIT_DIR = FINAL_DIR / "audit"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DIR / "foods_for_db.csv"
STAGE_FILE = PROCESSED_DIR / "usda_foods_stage.csv"

FINAL_FILE = FINAL_DIR / "foods_final.csv"

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_FILE)
print("Final output:", FINAL_FILE)


## 1. Configuration

These are product decisions, not hidden data-cleaning tricks.

For the current Spotter architecture, `Recipe` is separate from `Food`, so `MIXED_DISHES` is excluded by default.

You can change the flag later if you intentionally want things such as sandwiches, pizza, casseroles, etc. inside the generic food catalog.


In [ ]:
# Foods categorized as MIXED_DISHES behave more like prepared dishes/recipes.
EXCLUDE_MIXED_DISHES = True

# Source preference when multiple sources describe the same normalized food name.
SOURCE_PRIORITY = {
    "USDA_FDC_FOUNDATION": 1,
    "USDA_FDC_FNDDS": 2,
    "USDA_FDC_SR_LEGACY": 3,
}

# A duplicate group becomes a "conflict" audit group when the selected and
# non-selected rows differ by more than any of these values.
DUPLICATE_CONFLICT_TOLERANCE = {
    "calories_per_100g": 10.0,
    "protein_grams_per_100g": 2.0,
    "carbohydrate_grams_per_100g": 2.0,
    "fat_grams_per_100g": 2.0,
}

# Basic physical plausibility checks for nutrient values per 100g.
MAX_CALORIES_PER_100G = 1000.0
MAX_MACRO_GRAMS_PER_100G = 100.0


## 2. Load the normalized data


In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"{INPUT_FILE} does not exist. Run the first normalization notebook first."
    )

foods = pd.read_csv(INPUT_FILE, low_memory=False)

print("Rows loaded:", len(foods))
print("Columns:")
print(list(foods.columns))

display(foods.head(10))


## 3. Verify the Spotter database contract

We fail early if the first notebook did not produce the expected columns.


In [ ]:
DB_COLUMNS = [
    "name_en",
    "name_ar",
    "aliases",
    "category",
    "calories_per_100g",
    "protein_grams_per_100g",
    "carbohydrate_grams_per_100g",
    "fat_grams_per_100g",
    "created_by_user_id",
    "source",
    "external_source_id",
    "source_version",
    "is_active",
    "archived_at",
]

missing_columns = [column for column in DB_COLUMNS if column not in foods.columns]

if missing_columns:
    raise ValueError(
        "Input CSV is missing required Spotter columns: "
        + ", ".join(missing_columns)
    )

# Work only with the intended DB contract from this point onward.
foods = foods[DB_COLUMNS].copy()

print("✅ Input matches the Spotter Food contract.")


## 4. Helpers

The normalized-name key exists only to compare records.

Examples:

```text
"Chicken breast, grilled"
" chicken breast grilled "
"CHICKEN BREAST - GRILLED"
```

can collapse to the same comparison key.

The actual `name_en` stored in the final CSV remains unchanged.


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKC", str(value))
    value = re.sub(r"\s+", " ", value).strip()

    return value or None


def normalized_name_key(value):
    value = clean_text(value)

    if not value:
        return None

    value = value.casefold()
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value or None


def parse_nullable_bool(value):
    if pd.isna(value):
        return None

    if isinstance(value, bool):
        return value

    text = str(value).strip().casefold()

    if text in {"true", "1", "yes"}:
        return True

    if text in {"false", "0", "no"}:
        return False

    return None


## 5. Standardize types without changing meaning


In [ ]:
TEXT_COLUMNS = [
    "name_en",
    "name_ar",
    "category",
    "created_by_user_id",
    "source",
    "external_source_id",
    "source_version",
    "archived_at",
]

for column in TEXT_COLUMNS:
    foods[column] = foods[column].map(clean_text)

NUTRITION_COLUMNS = [
    "calories_per_100g",
    "protein_grams_per_100g",
    "carbohydrate_grams_per_100g",
    "fat_grams_per_100g",
]

for column in NUTRITION_COLUMNS:
    foods[column] = pd.to_numeric(foods[column], errors="coerce").round(2)

foods["is_active"] = foods["is_active"].map(parse_nullable_bool).fillna(True)

# Global imported USDA foods are not user-created.
foods["created_by_user_id"] = None

# We are not adding Arabic names yet.
foods["name_ar"] = None

# We are not inventing aliases in this pipeline.
foods["aliases"] = None

foods["_name_key"] = foods["name_en"].map(normalized_name_key)
foods["_source_priority"] = foods["source"].map(SOURCE_PRIORITY).fillna(99).astype(int)

display(foods.head(10))


## 6. Remove rows that should never enter the database

This is stricter than EDA.

A final database row must have:

- an English name
- name length <= 200 characters
- all four required nutrition values
- non-negative nutrition
- calories <= 1000 kcal / 100g
- protein/carbohydrate/fat <= 100g / 100g
- source
- external source ID


In [ ]:
def get_invalid_reason(row):
    reasons = []

    name = row["name_en"]

    if not name:
        reasons.append("missing_name")
    elif len(name) > 200:
        reasons.append("name_over_200_chars")

    for column in NUTRITION_COLUMNS:
        value = row[column]

        if pd.isna(value):
            reasons.append(f"missing_{column}")
        elif value < 0:
            reasons.append(f"negative_{column}")

    calories = row["calories_per_100g"]

    if pd.notna(calories) and calories > MAX_CALORIES_PER_100G:
        reasons.append("calories_over_1000_per_100g")

    for column in [
        "protein_grams_per_100g",
        "carbohydrate_grams_per_100g",
        "fat_grams_per_100g",
    ]:
        value = row[column]

        if pd.notna(value) and value > MAX_MACRO_GRAMS_PER_100G:
            reasons.append(f"{column}_over_100")

    if not row["source"]:
        reasons.append("missing_source")

    if not row["external_source_id"]:
        reasons.append("missing_external_source_id")

    if not row["_name_key"]:
        reasons.append("missing_normalized_name_key")

    return "|".join(reasons)


foods["_invalid_reason"] = foods.apply(get_invalid_reason, axis=1)

invalid_rows = foods[foods["_invalid_reason"] != ""].copy()
valid = foods[foods["_invalid_reason"] == ""].copy()

print("Valid before product filtering:", len(valid))
print("Invalid:", len(invalid_rows))

display(
    invalid_rows[
        ["name_en", "source", "external_source_id", "_invalid_reason"]
    ].head(30)
)


## 7. Remove mixed dishes from the Food catalog

Because Spotter now has:

```text
Food
Recipe
RecipeIngredient
```

prepared mixed dishes belong more naturally in `Recipe`.

This step is configurable and fully audited.


In [ ]:
if EXCLUDE_MIXED_DISHES:
    mixed_mask = valid["category"].eq("MIXED_DISHES")

    dropped_mixed_dishes = valid[mixed_mask].copy()
    valid = valid[~mixed_mask].copy()
else:
    dropped_mixed_dishes = valid.iloc[0:0].copy()

print("Dropped MIXED_DISHES:", len(dropped_mixed_dishes))
print("Remaining:", len(valid))

display(
    dropped_mixed_dishes[
        ["name_en", "category", "source", "external_source_id"]
    ].head(30)
)


## 8. Enforce source identity uniqueness first

Your Prisma schema contains:

```prisma
@@unique([source, externalSourceId])
```

So if the input contains the same source identity more than once, one deterministic row must remain.

This is different from user-facing name deduplication.


In [ ]:
source_id_duplicate_mask = valid.duplicated(
    subset=["source", "external_source_id"],
    keep=False,
)

source_id_duplicate_rows = valid[source_id_duplicate_mask].copy()

if not source_id_duplicate_rows.empty:
    print(
        "⚠️ Duplicate (source, external_source_id) rows found:",
        len(source_id_duplicate_rows)
    )

    display(
        source_id_duplicate_rows[
            [
                "name_en",
                "source",
                "external_source_id",
                "calories_per_100g",
            ]
        ].head(50)
    )

# Deterministic: sort first, then keep one per source identity.
valid = (
    valid
    .sort_values(
        ["source", "external_source_id", "name_en"],
        kind="stable",
    )
    .drop_duplicates(
        subset=["source", "external_source_id"],
        keep="first",
    )
    .reset_index(drop=True)
)

print("Rows after source-ID uniqueness:", len(valid))


## 9. Detect exact normalized-name duplicate groups

Now we are solving the actual user-facing problem:

```text
search "egg"
→ Egg, whole, raw
→ Egg, whole, raw
```

If two rows normalize to the same name, Spotter keeps one catalog record.


In [ ]:
name_duplicate_mask = valid.duplicated("_name_key", keep=False)

name_duplicate_rows = (
    valid[name_duplicate_mask]
    .sort_values(
        ["_name_key", "_source_priority", "source", "external_source_id"],
        kind="stable",
    )
    .copy()
)

duplicate_group_count = name_duplicate_rows["_name_key"].nunique()

print("Duplicate name groups:", duplicate_group_count)
print("Rows inside duplicate groups:", len(name_duplicate_rows))

display(
    name_duplicate_rows[
        [
            "name_en",
            "source",
            "external_source_id",
            "category",
            "calories_per_100g",
            "protein_grams_per_100g",
            "carbohydrate_grams_per_100g",
            "fat_grams_per_100g",
        ]
    ].head(60)
)


## 10. Flag duplicate groups whose nutrition disagrees

This does **not** block final export.

It gives you an audit file containing same-name groups where USDA sources disagree enough that you may later want to inspect them.

The final winner is still selected using the explicit source priority.


In [ ]:
def duplicate_group_has_conflict(group):
    if len(group) <= 1:
        return False

    preferred = group.sort_values(
        ["_source_priority", "source", "external_source_id"],
        kind="stable",
    ).iloc[0]

    for _, candidate in group.iterrows():
        if candidate.name == preferred.name:
            continue

        for column, tolerance in DUPLICATE_CONFLICT_TOLERANCE.items():
            preferred_value = preferred[column]
            candidate_value = candidate[column]

            if pd.isna(preferred_value) or pd.isna(candidate_value):
                continue

            if abs(float(preferred_value) - float(candidate_value)) > tolerance:
                return True

    return False


conflicting_keys = []

for name_key, group in name_duplicate_rows.groupby("_name_key", sort=False):
    if duplicate_group_has_conflict(group):
        conflicting_keys.append(name_key)

duplicate_conflicts = name_duplicate_rows[
    name_duplicate_rows["_name_key"].isin(conflicting_keys)
].copy()

print("Duplicate groups with meaningful nutrition disagreement:", len(conflicting_keys))
print("Rows in conflict audit:", len(duplicate_conflicts))

display(
    duplicate_conflicts[
        [
            "name_en",
            "source",
            "external_source_id",
            "calories_per_100g",
            "protein_grams_per_100g",
            "carbohydrate_grams_per_100g",
            "fat_grams_per_100g",
        ]
    ].head(60)
)


## 11. Select one winner per normalized food name

Selection policy:

```text
Foundation
    ↓
FNDDS
    ↓
SR Legacy
```

Within the same source, the external source ID creates deterministic ordering.

If the preferred row has `OTHER`/missing category and another duplicate has a more useful category, the winner inherits that category. Nutrition and source identity are **never mixed** between records.


In [ ]:
def choose_best_category(group, winner_category):
    if winner_category and winner_category != "OTHER":
        return winner_category

    candidates = [
        value
        for value in group["category"].tolist()
        if value and value != "OTHER"
    ]

    if candidates:
        return candidates[0]

    return winner_category


selected_rows = []
dropped_rows = []

for name_key, group in valid.groupby("_name_key", sort=False):
    ordered = group.sort_values(
        ["_source_priority", "source", "external_source_id"],
        kind="stable",
    )

    winner = ordered.iloc[0].copy()

    winner["category"] = choose_best_category(
        ordered,
        winner["category"],
    )

    selected_rows.append(winner)

    if len(ordered) > 1:
        losers = ordered.iloc[1:].copy()
        losers["_kept_source"] = winner["source"]
        losers["_kept_external_source_id"] = winner["external_source_id"]
        losers["_kept_name_en"] = winner["name_en"]

        dropped_rows.append(losers)


final = pd.DataFrame(selected_rows).reset_index(drop=True)

if dropped_rows:
    dropped_exact_name_duplicates = pd.concat(
        dropped_rows,
        ignore_index=True,
    )
else:
    dropped_exact_name_duplicates = valid.iloc[0:0].copy()

print("Rows before name deduplication:", len(valid))
print("Duplicates removed:", len(dropped_exact_name_duplicates))
print("Rows after name deduplication:", len(final))


## 12. Final database invariants

Before export, prove that the resulting data satisfies the assumptions behind the Prisma model.


In [ ]:
# 1. Unique external source identity
assert not final.duplicated(
    subset=["source", "external_source_id"]
).any(), "Duplicate (source, external_source_id) remained."

# 2. One normalized user-facing name
assert not final.duplicated("_name_key").any(), (
    "Duplicate normalized food names remained."
)

# 3. Required nutrition exists
assert not final[NUTRITION_COLUMNS].isna().any().any(), (
    "Missing required nutrition remained."
)

# 4. Nutrition is non-negative
assert (final[NUTRITION_COLUMNS] >= 0).all().all(), (
    "Negative nutrition remained."
)

# 5. Names fit Prisma VarChar(200)
assert final["name_en"].map(len).le(200).all(), (
    "A name still exceeds 200 characters."
)

# 6. Imported rows are global foods
assert final["created_by_user_id"].isna().all(), (
    "Imported USDA foods must have created_by_user_id = NULL."
)

print("✅ All final invariants passed.")


## 13. Optional calorie-vs-macro QA

This does not reject rows.

A rough energy estimate is:

```text
protein × 4 + carbohydrate × 4 + fat × 9
```

But USDA calories can legitimately differ because of fiber, alcohol, specific Atwater factors, rounding, etc.

This section only helps spot extreme surprises.


In [ ]:
final["_macro_estimated_calories"] = (
    final["protein_grams_per_100g"] * 4
    + final["carbohydrate_grams_per_100g"] * 4
    + final["fat_grams_per_100g"] * 9
)

final["_calorie_difference"] = (
    final["calories_per_100g"] - final["_macro_estimated_calories"]
).abs()

print("Largest calorie-vs-macro differences:")

display(
    final.sort_values(
        "_calorie_difference",
        ascending=False,
    )[
        [
            "name_en",
            "calories_per_100g",
            "_macro_estimated_calories",
            "_calorie_difference",
            "source",
        ]
    ].head(30)
)


## 14. Build the exact final CSV

The final file intentionally contains only fields relevant to your Prisma `Food` insert.

Database-generated fields are omitted:

```text
id
created_at
updated_at
```


In [ ]:
FINAL_DB_COLUMNS = [
    "name_en",
    "name_ar",
    "aliases",
    "category",
    "calories_per_100g",
    "protein_grams_per_100g",
    "carbohydrate_grams_per_100g",
    "fat_grams_per_100g",
    "created_by_user_id",
    "source",
    "external_source_id",
    "source_version",
    "is_active",
    "archived_at",
]

foods_final = final[FINAL_DB_COLUMNS].copy()

# Stable final ordering makes Git diffs and future imports easier to inspect.
foods_final = (
    foods_final
    .sort_values(
        ["category", "name_en", "source", "external_source_id"],
        na_position="last",
        kind="stable",
    )
    .reset_index(drop=True)
)

print("FINAL DB ROW COUNT:", len(foods_final))
display(foods_final.head(30))


## 15. Write the final CSV and audit files


In [ ]:
INVALID_AUDIT = AUDIT_DIR / "dropped_invalid_rows.csv"
MIXED_AUDIT = AUDIT_DIR / "dropped_mixed_dishes.csv"
DUPLICATE_AUDIT = AUDIT_DIR / "dropped_exact_name_duplicates.csv"
CONFLICT_AUDIT = AUDIT_DIR / "duplicate_conflicts.csv"

foods_final.to_csv(FINAL_FILE, index=False)

invalid_rows.to_csv(INVALID_AUDIT, index=False)
dropped_mixed_dishes.to_csv(MIXED_AUDIT, index=False)
dropped_exact_name_duplicates.to_csv(DUPLICATE_AUDIT, index=False)
duplicate_conflicts.to_csv(CONFLICT_AUDIT, index=False)

print("✅ FINAL CSV:")
print("  ", FINAL_FILE)

print("\nAudit files:")
print("  ", INVALID_AUDIT)
print("  ", MIXED_AUDIT)
print("  ", DUPLICATE_AUDIT)
print("  ", CONFLICT_AUDIT)


## 16. Final report


In [ ]:
report = pd.DataFrame({
    "metric": [
        "input processed rows",
        "invalid rows removed",
        "mixed dishes removed",
        "exact-name duplicate rows removed",
        "duplicate conflict groups",
        "FINAL DATABASE ROWS",
    ],
    "count": [
        len(foods),
        len(invalid_rows),
        len(dropped_mixed_dishes),
        len(dropped_exact_name_duplicates),
        len(conflicting_keys),
        len(foods_final),
    ],
})

display(report)

print("\nRows by source:")
display(
    foods_final["source"]
    .value_counts(dropna=False)
    .rename_axis("source")
    .to_frame("rows")
)

print("\nRows by Spotter category:")
display(
    foods_final["category"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .to_frame("rows")
)

print("\nRandom final sample:")
display(
    foods_final.sample(
        min(30, len(foods_final)),
        random_state=42,
    )
)


# Result

If every assertion passes, the file to import into Spotter is:

```text
data/final/foods_final.csv
```

Do not import the audit files.

The next application step is a **Node + Prisma importer** that reads `foods_final.csv` and performs an idempotent upsert using:

```prisma
@@unique([source, externalSourceId])
```

That importer should remain separate from the EDA/normalization notebooks.
